In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import random
from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Flatten, Dense, Conv2D, MaxPooling2D
from tensorflow.keras.optimizers import SGD
from concurrent.futures import ThreadPoolExecutor

In [2]:
tf.config.threading.set_intra_op_parallelism_threads(1)
tf.config.threading.set_inter_op_parallelism_threads(1)
print("Environment set. TF restricted to 1 thread per instance to allow concurrency.")

Environment set. TF restricted to 1 thread per instance to allow concurrency.


In [3]:
# Loads the MNIST dataset and normalizes 
(X_train, Y_train), (X_test, Y_test)=mnist.load_data()
X_train=X_train.astype(np.float32)/255.0
X_test=X_test.astype(np.float32)/255.0
X_train=X_train.reshape(-1, 28, 28, 1)
X_test=X_test.reshape(-1, 28, 28, 1)

In [4]:
def iid_partition(x, y, num_clients):
    idx=np.random.permutation(len(x))
    x_shuffled, y_shuffled=x[idx], y[idx]
    x_parts=np.array_split(x_shuffled, num_clients)
    y_parts=np.array_split(y_shuffled, num_clients)
    partitions=list(zip(x_parts, y_parts))
    return partitions

In [5]:
def non_iid_partition(x, y, num_clients):
    sorted_idx=np.argsort(y)
    X_sorted, Y_sorted=x[sorted_idx], y[sorted_idx]
    num_shards=2*num_clients
    size_shards=len(x)//num_shards
    shards_idx=np.random.permutation(num_shards)
    partitions=[]
    for i in range(num_clients):
        s1, s2= shards_idx[2*i], shards_idx[2*i+1]
        X_client=np.concatenate((X_sorted[s1*size_shards:(s1+1)*size_shards], X_sorted[s2*size_shards:(s2+1)*size_shards]), axis=0)
        Y_client=np.concatenate((Y_sorted[s1*size_shards:(s1+1)*size_shards], Y_sorted[s2*size_shards:(s2+1)*size_shards]), axis=0)
        idx=np.random.permutation(len(X_client))
        partitions.append((X_client[idx], Y_client[idx]))
    return partitions

In [6]:
def cnn_model():
    model=Sequential([
        Input(shape=(28, 28, 1)),
        Conv2D(32, kernel_size=(3, 3), activation='relu'),
        MaxPooling2D(pool_size=(2, 2)),
        Conv2D(64, kernel_size=(3, 3), activation='relu'),
        MaxPooling2D(pool_size=(2, 2)),
        Flatten(),
        Dense(128, activation='relu'),
        Dense(10, activation='softmax'),
    ])
    model.compile(optimizer=SGD(learning_rate=0.01), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

In [7]:
# Trains the model locally on a client, modifying gradients using SCAFFOLD control variates
def local_training_scaffold(client_data, global_weights, global_c, local_c, local_epochs=2, lr=0.01):
    X_local, Y_local=client_data
    
    model=cnn_model()
    model.set_weights(global_weights)
    optimizer=SGD(learning_rate=lr)
    loss_fn=tf.keras.losses.SparseCategoricalCrossentropy()

    global_c_tensors=[tf.constant(c, dtype=tf.float32) for c in global_c]
    local_c_tensors=[tf.constant(c, dtype=tf.float32) for c in local_c]
    
    dataset=tf.data.Dataset.from_tensor_slices((X_local, Y_local)).batch(32)
    total_steps=0
    
    def train_step(x, y):
        with tf.GradientTape() as tape:
            y_pred=model(x, training=True)
            loss=loss_fn(y, y_pred)
            
        grads=tape.gradient(loss, model.trainable_variables)
        # Applies variance reduction correction: adjusted_grad = grad - local_c + global_c
        adjusted_grads=[]
        for g, c_i, c_g in zip(grads, local_c_tensors, global_c_tensors):
            adjusted_grads.append(g - c_i + c_g)
            
        optimizer.apply_gradients(zip(adjusted_grads, model.trainable_variables))
        
    for epoch in range(local_epochs):
        for batch_x, batch_y in dataset:
            train_step(batch_x, batch_y)
            total_steps += 1
    # Updates the client's local control variates state      
    local_weights=model.get_weights()
    delta_y=[]
    delta_c=[]
    new_local_c=[]
    
    for i in range(len(global_weights)):
        dy=local_weights[i]-global_weights[i]
        delta_y.append(dy)
        
        c_plus=local_c[i]-global_c[i]-(dy/(total_steps*lr))
        
        dc=c_plus-local_c[i]
        
        delta_c.append(dc)
        new_local_c.append(c_plus)
        
    return delta_y, delta_c, new_local_c

In [8]:
# Averages active client updates to refresh central weights and global control variates
def aggregate_updates_scaffold(client_updates, global_weights, global_c, num_total_clients, global_lr=1.0):
    num_active=len(client_updates)
    
    new_weights=[w.copy() for w in global_weights]
    new_global_c=[c.copy() for c in global_c]
    
    for i in range(len(new_weights)):
        sum_dy=sum(update['dy'][i] for update in client_updates)
        avg_dy=sum_dy/num_active
        new_weights[i]+=global_lr*avg_dy
        
        sum_dc=sum(update['dc'][i] for update in client_updates)
        new_global_c[i]+=sum_dc / num_total_clients
        
    return new_weights, new_global_c

In [9]:
def evaluate_global(global_model, X_test, Y_test):
    loss, accuracy = global_model.evaluate(X_test, Y_test, verbose=1)
    return loss, accuracy

In [10]:
def run_scaffold_simulation(partitions, X_test, Y_test, num_clients, rounds, participation_rate):
    global_model=cnn_model()
    global_weights=global_model.get_weights()
    
    global_c=[np.zeros_like(w) for w in global_weights]
    
    client_local_c={i: [np.zeros_like(w) for w in global_weights] for i in range(num_clients)}
    
    max_workers=min(num_clients, 12)
    history=[]
    
    for r in range(1, rounds + 1):
        num_active=max(1, int(num_clients * participation_rate))
        active_clients=random.sample(range(num_clients), num_active)
        client_updates=[]
        
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures={
                executor.submit(local_training_scaffold, partitions[i], global_weights, global_c, client_local_c[i]): i for i in active_clients
            }
            
            for future in futures:
                client_idx=futures[future]
                dy, dc, new_c_i=future.result()
                
                client_local_c[client_idx]=new_c_i
                client_updates.append({'dy':dy, 'dc':dc})
                
        global_weights, global_c=aggregate_updates_scaffold(client_updates, global_weights, global_c, num_clients)
        global_model.set_weights(global_weights)
        
        loss, acc=evaluate_global(global_model, X_test, Y_test)
        history.append({'round': r, 'loss': loss, 'acc': acc})
        print(f"Round {r} | Accuracy: {acc} | Loss: {loss}")
        
    return history

In [11]:
CLIENTS = 10
IID_ROUNDS = 20
PARTICIPATION_RATE = 1.0

print(f"=== RUNNING IID SCAFFOLD ({CLIENTS} Clients, {PARTICIPATION_RATE*100}% Participation) ===")
iid_parts = iid_partition(X_train, Y_train, CLIENTS)
iid_history1 = run_scaffold_simulation(iid_parts, X_test, Y_test, CLIENTS, IID_ROUNDS, PARTICIPATION_RATE)

NON_IID_ROUNDS=50
print(f"\n=== RUNNING NON-IID SCAFFOLD ({CLIENTS} Clients, {PARTICIPATION_RATE*100}% Participation) ===")
non_iid_parts = non_iid_partition(X_train, Y_train, CLIENTS)
non_iid_history1 = run_scaffold_simulation(non_iid_parts, X_test, Y_test, CLIENTS, NON_IID_ROUNDS, PARTICIPATION_RATE)

=== RUNNING IID SCAFFOLD (10 Clients, 100.0% Participation) ===
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - accuracy: 0.8543 - loss: 0.5105
Round 1 | Accuracy: 0.8543000221252441 | Loss: 0.5104854702949524
313/313 ━━━━━━━━━━━━━━━━━━━━ 5s 15ms/step - accuracy: 0.9090 - loss: 0.3014
Round 2 | Accuracy: 0.9089999794960022 | Loss: 0.3013899326324463
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - accuracy: 0.9304 - loss: 0.2298
Round 3 | Accuracy: 0.930400013923645 | Loss: 0.22980976104736328
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 14ms/step - accuracy: 0.9448 - loss: 0.1856
Round 4 | Accuracy: 0.9448000192642212 | Loss: 0.18563978374004364
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 14ms/step - accuracy: 0.9553 - loss: 0.1536
Round 5 | Accuracy: 0.955299973487854 | Loss: 0.1535901427268982
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - accuracy: 0.9626 - loss: 0.1308
Round 6 | Accuracy: 0.9625999927520752 | Loss: 0.13084158301353455
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 14ms/step - accuracy: 0.9674 - loss: 0.1145
Round 7

In [13]:
IID_ROUNDS=50
PARTICIPATION_RATE=0.5
print(f"===RUNNING IID FEDERATED LEARNING 50% PARTICIPATION({CLIENTS} Clients)===")
iid_history2=run_scaffold_simulation(iid_parts,X_test,Y_test,CLIENTS,IID_ROUNDS,PARTICIPATION_RATE)

NON_IID_ROUNDS=100
print(f"===RUNNING NON-IID FEDERATED LEARNING 50% PARTICIPATION({CLIENTS} Clients)===")
non_iid_history2=run_scaffold_simulation(non_iid_parts,X_test,Y_test,CLIENTS,NON_IID_ROUNDS,PARTICIPATION_RATE)

===RUNNING IID FEDERATED LEARNING 50% PARTICIPATION(10 Clients)===
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - accuracy: 0.8819 - loss: 0.4248
Round 1 | Accuracy: 0.8819000124931335 | Loss: 0.4247751832008362
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - accuracy: 0.9174 - loss: 0.2830
Round 2 | Accuracy: 0.9174000024795532 | Loss: 0.28302454948425293
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - accuracy: 0.9350 - loss: 0.2109
Round 3 | Accuracy: 0.9350000023841858 | Loss: 0.21092341840267181
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - accuracy: 0.9482 - loss: 0.1730
Round 4 | Accuracy: 0.948199987411499 | Loss: 0.17303432524204254
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - accuracy: 0.9583 - loss: 0.1462
Round 5 | Accuracy: 0.958299994468689 | Loss: 0.14616093039512634
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - accuracy: 0.9652 - loss: 0.1247
Round 6 | Accuracy: 0.9652000069618225 | Loss: 0.12465683370828629
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - accuracy: 0.9670 - loss: 0.1094
Ro

In [14]:
IID_ROUNDS=50
PARTICIPATION_RATE=0.3
print(f"===RUNNING IID FEDERATED LEARNING 30% PARTICIPATION({CLIENTS} Clients)===")
iid_history3=run_scaffold_simulation(iid_parts,X_test,Y_test,CLIENTS,IID_ROUNDS,PARTICIPATION_RATE)
NON_IID_ROUNDS=100
print(f"===RUNNING NON-IID FEDERATED LEARNING 30% PARTICIPATION({CLIENTS} Clients)===")
non_iid_history3=run_scaffold_simulation(non_iid_parts,X_test,Y_test,CLIENTS,NON_IID_ROUNDS,PARTICIPATION_RATE)

===RUNNING IID FEDERATED LEARNING 30% PARTICIPATION(10 Clients)===
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - accuracy: 0.8570 - loss: 0.5339
Round 1 | Accuracy: 0.8569999933242798 | Loss: 0.5339478254318237
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - accuracy: 0.9062 - loss: 0.3068
Round 2 | Accuracy: 0.9061999917030334 | Loss: 0.306790828704834
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - accuracy: 0.9299 - loss: 0.2497
Round 3 | Accuracy: 0.9298999905586243 | Loss: 0.24968579411506653
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - accuracy: 0.9245 - loss: 0.2348
Round 4 | Accuracy: 0.9244999885559082 | Loss: 0.234833225607872
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step - accuracy: 0.9543 - loss: 0.1662
Round 5 | Accuracy: 0.9542999863624573 | Loss: 0.16616611182689667
313/313 ━━━━━━━━━━━━━━━━━━━━ 12s 37ms/step - accuracy: 0.9631 - loss: 0.1361
Round 6 | Accuracy: 0.963100016117096 | Loss: 0.13605159521102905
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - accuracy: 0.9637 - loss: 0.1221
Roun

In [15]:
np.save("scaffold_iid_history1.npy", iid_history1)
np.save("scaaffold_non_iid_history1.npy", non_iid_history1)

np.save("scaffold_iid_history2.npy", iid_history2)
np.save("scaffold_non_iid_history2.npy", non_iid_history2)

np.save("scaffold_iid_history3.npy", iid_history3)
np.save("scaffold_non_iid_history3.npy", non_iid_history3)